# TP4: From UDF to panda UDF (according to spark)

PySpark UDF (a.k.a User Defined Function) is one of the most useful features of Spark SQL & DataFrame which is used to extend PySpark's build capabilities.

1. Introduction to UDF

2. Introduction to Pandas UDF

### 1. Start a spark session

(we want to start a local session and give it a name).

We'll use the SparkSession class from the *pyspark.sql* sub-module.

Call *sc* the *spark.sparkContext*.

- Indicate what is defined by *SparkSession* and *sparkContext*.

- the master is 'local' (specify the number of cores) and the application must be given a name.

In [1]:
from datetime import datetime, date
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

from pyspark.sql.functions import col 
from pyspark.sql import Row

import findspark
findspark.init()

In [2]:
spark = SparkSession.builder.master("local[*]").appName('TP4_UDF').getOrCreate()
sc = spark.sparkContext

In [3]:
print("pyspark version:",pyspark.__version__)
sc

pyspark version: 4.0.1


<SparkContext master=local[*] appName=TP4_UDF>

A port (4040) that you should definitely look at.

It allows you to follow the progress of a spark process.

http://localhost:4040

## 1. User Defined Function¶

User-Defined Functions (UDFs) are user-programmable routines that act on one row. 

Note: UDFs are the most expensive operations, so only use them when you have no choice.

PySpark UDF's are similar to UDF on traditional databases. In PySpark, you create a function in a Python syntax and wrap it with PySpark SQL udf() or register it as udf and use it on DataFrame and SQL respectively.

Before you create any UDF, do your research to check if the similar function you wanted is already available in Spark SQL Functions. PySpark SQL provides several predefined common functions and many more new functions are added with every release.

Now convert this function to UDF by passing the function to PySpark SQL udf(), this function is available at org.apache.spark.sql.functions.udf package. 

Make sure you import this package before using it.

PySpark SQL udf() function returns org.apache.spark.sql.expressions.UserDefinedFunction class object.

pyspark.sql.functions.udf

[udf function]( https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.udf.html?highlight=udf#pyspark.sql.functions.udf)

### 1.1. Les imports

 

In [4]:
# import udf
from pyspark.sql.functions import udf
# import des Types
from pyspark.sql.types import IntegerType, FloatType

In order to use convertCase() function on PySpark SQL, you need to register the function with PySpark by using spark.udf.register().



### 1.2 Creating a data frame

- create 2 data frames with 2 columns (label, value)

The difference between the 2 dataframes is that for the first, the "value" values are of type long or int, and for the second, the "value" values are of type double.

The label can take 2 values and appears in at least 4 lines.

In [5]:
# DataFrame with int for value column
l1 = [(1, 0), (1, 8), (1, 3), (1, 3), (2, 1), (2, 9), (2, 6), (2, 6), (2, 11), (2, 2)]
df1 = spark.createDataFrame(l1,['label', 'value'])
# Display schema and dataframe
df1.printSchema()
df1.show()

# DataFrame with double for value column
l2 = [(1, 0.), (1, 8.), (1, 3.), (1, 3.), (2, 1.), (2, 9.), (2, 6.), (2, 6.), (2, 11.), (2, 2.)]
df2 = spark.createDataFrame(l2,['label', 'value'])
# Display schela and dataframe
df2.printSchema()
df2.show()

root
 |-- label: long (nullable = true)
 |-- value: long (nullable = true)

+-----+-----+
|label|value|
+-----+-----+
|    1|    0|
|    1|    8|
|    1|    3|
|    1|    3|
|    2|    1|
|    2|    9|
|    2|    6|
|    2|    6|
|    2|   11|
|    2|    2|
+-----+-----+

root
 |-- label: long (nullable = true)
 |-- value: double (nullable = true)

+-----+-----+
|label|value|
+-----+-----+
|    1|  0.0|
|    1|  8.0|
|    1|  3.0|
|    1|  3.0|
|    2|  1.0|
|    2|  9.0|
|    2|  6.0|
|    2|  6.0|
|    2| 11.0|
|    2|  2.0|
+-----+-----+



### 1.3. How to use UDF in data frames?¶

- Creating udf functions

We'll start by creating a python function that squares a number.

Convert the python function into a udf function using the *udf* function

the first udf function (squareUDF_int) will have an output of type IntegerType()

the second function (squareUDF_float) will have an output of type IntegerType()

In [6]:
# python function
def squared1(s):
  return s * s

# Converting function to UDF 
squareUDF_int = udf(lambda z: squared1(z),IntegerType())

# Converting function to UDF 
squareUDF_float = udf(lambda z: squared1(z),FloatType())

apply squareUDF_int and squareUDF_float to both dataframes.

We'll start by using *select* from the dataframe and then *withColumn*.

- using *select* and *col*.


In [7]:
# using squareUDF_int and squareUDF_float on df1
df1.select(col("label"), col("value"), squareUDF_int(col("value")).alias("squared")).show()

df1.select(col("label"), col("value"), squareUDF_float(col("value")).alias("squared")).show()

# using squareUDF_int and squareUDF_float on df2
df2.select(col("label"), col("value"), squareUDF_int(col("value")).alias("squared")).show()

df2.select(col("label"), col("value"), squareUDF_float(col("value")).alias("squared")).show()

+-----+-----+-------+
|label|value|squared|
+-----+-----+-------+
|    1|    0|      0|
|    1|    8|     64|
|    1|    3|      9|
|    1|    3|      9|
|    2|    1|      1|
|    2|    9|     81|
|    2|    6|     36|
|    2|    6|     36|
|    2|   11|    121|
|    2|    2|      4|
+-----+-----+-------+

+-----+-----+-------+
|label|value|squared|
+-----+-----+-------+
|    1|    0|   NULL|
|    1|    8|   NULL|
|    1|    3|   NULL|
|    1|    3|   NULL|
|    2|    1|   NULL|
|    2|    9|   NULL|
|    2|    6|   NULL|
|    2|    6|   NULL|
|    2|   11|   NULL|
|    2|    2|   NULL|
+-----+-----+-------+

+-----+-----+-------+
|label|value|squared|
+-----+-----+-------+
|    1|  0.0|   NULL|
|    1|  8.0|   NULL|
|    1|  3.0|   NULL|
|    1|  3.0|   NULL|
|    2|  1.0|   NULL|
|    2|  9.0|   NULL|
|    2|  6.0|   NULL|
|    2|  6.0|   NULL|
|    2| 11.0|   NULL|
|    2|  2.0|   NULL|
+-----+-----+-------+

+-----+-----+-------+
|label|value|squared|
+-----+-----+-------+
|    1|

- Comment and analyze

- using *withColumn* and *col*

In [8]:
# using squareUDF_int and squareUDF_float on df1
df1.withColumn("squared1",squareUDF_int(col("value"))).show()

df1.withColumn("squared1",squareUDF_float(col("value"))).show()

# using squareUDF_int and squareUDF_float on df2
df2.withColumn("squared1",squareUDF_int(col("value"))).show()

df2.withColumn("squared1",squareUDF_float(col("value"))).show()

+-----+-----+--------+
|label|value|squared1|
+-----+-----+--------+
|    1|    0|       0|
|    1|    8|      64|
|    1|    3|       9|
|    1|    3|       9|
|    2|    1|       1|
|    2|    9|      81|
|    2|    6|      36|
|    2|    6|      36|
|    2|   11|     121|
|    2|    2|       4|
+-----+-----+--------+

+-----+-----+--------+
|label|value|squared1|
+-----+-----+--------+
|    1|    0|    NULL|
|    1|    8|    NULL|
|    1|    3|    NULL|
|    1|    3|    NULL|
|    2|    1|    NULL|
|    2|    9|    NULL|
|    2|    6|    NULL|
|    2|    6|    NULL|
|    2|   11|    NULL|
|    2|    2|    NULL|
+-----+-----+--------+

+-----+-----+--------+
|label|value|squared1|
+-----+-----+--------+
|    1|  0.0|    NULL|
|    1|  8.0|    NULL|
|    1|  3.0|    NULL|
|    1|  3.0|    NULL|
|    2|  1.0|    NULL|
|    2|  9.0|    NULL|
|    2|  6.0|    NULL|
|    2|  6.0|    NULL|
|    2| 11.0|    NULL|
|    2|  2.0|    NULL|
+-----+-----+--------+

+-----+-----+--------+
|label|v

- Comment and analyze

### 1.4 Registering PySpark UDF & use it on SQL

In order to use convertCase() function on PySpark SQL, you need to register the function with PySpark by using spark.udf.register().

Using *createOrReplaceTempView* to create or replace a local temporary view with this DataFrame.

Write and use an sql command selecting the label and value columns and adding a third column by squaring the value column.


In [9]:
from pyspark.sql.types import LongType

#python function
def squared2(s):
  return s * s

# registering
spark.udf.register("square2r_int", squared2, LongType())
spark.udf.register("square2r_float", squared2, FloatType())

# defining view
df1.createOrReplaceTempView("VAL_TABLE_1")
df2.createOrReplaceTempView("VAL_TABLE_2")

# application
spark.sql("select label, value, square2r_int(value) as Square from VAL_TABLE_1") \
     .show()

spark.sql("select label, value, square2r_float(value) as Square from VAL_TABLE_1") \
     .show()

spark.sql("select label, value, square2r_int(value) as Square from VAL_TABLE_2") \
     .show()

spark.sql("select label, value, square2r_float(value) as Square from VAL_TABLE_2") \
     .show()

+-----+-----+------+
|label|value|Square|
+-----+-----+------+
|    1|    0|     0|
|    1|    8|    64|
|    1|    3|     9|
|    1|    3|     9|
|    2|    1|     1|
|    2|    9|    81|
|    2|    6|    36|
|    2|    6|    36|
|    2|   11|   121|
|    2|    2|     4|
+-----+-----+------+

+-----+-----+------+
|label|value|Square|
+-----+-----+------+
|    1|    0|  NULL|
|    1|    8|  NULL|
|    1|    3|  NULL|
|    1|    3|  NULL|
|    2|    1|  NULL|
|    2|    9|  NULL|
|    2|    6|  NULL|
|    2|    6|  NULL|
|    2|   11|  NULL|
|    2|    2|  NULL|
+-----+-----+------+

+-----+-----+------+
|label|value|Square|
+-----+-----+------+
|    1|  0.0|  NULL|
|    1|  8.0|  NULL|
|    1|  3.0|  NULL|
|    1|  3.0|  NULL|
|    2|  1.0|  NULL|
|    2|  9.0|  NULL|
|    2|  6.0|  NULL|
|    2|  6.0|  NULL|
|    2| 11.0|  NULL|
|    2|  2.0|  NULL|
+-----+-----+------+

+-----+-----+------+
|label|value|Square|
+-----+-----+------+
|    1|  0.0|   0.0|
|    1|  8.0|  64.0|
|    1|  3

### 1.5 Creating UDF using annotation (decorator ?)

In the previous sections, you have learned creating a UDF is a 2 step process, first, you need to create a Python function, second convert function to UDF using SQL udf() function, however, you can avoid these two steps and create it with just a single step by using annotations.


In [10]:
@udf("long")
def square3(s):
  return s * s

@udf("float")
def square4(s):
  return s * s

df1.withColumn("squared3",square3(col("value"))).show()

df1.withColumn("squared3",square4(col("value"))).show()

df2.withColumn("squared3",square3(col("value"))).show()

df2.withColumn("squared3",square4(col("value"))).show()

+-----+-----+--------+
|label|value|squared3|
+-----+-----+--------+
|    1|    0|       0|
|    1|    8|      64|
|    1|    3|       9|
|    1|    3|       9|
|    2|    1|       1|
|    2|    9|      81|
|    2|    6|      36|
|    2|    6|      36|
|    2|   11|     121|
|    2|    2|       4|
+-----+-----+--------+

+-----+-----+--------+
|label|value|squared3|
+-----+-----+--------+
|    1|    0|    NULL|
|    1|    8|    NULL|
|    1|    3|    NULL|
|    1|    3|    NULL|
|    2|    1|    NULL|
|    2|    9|    NULL|
|    2|    6|    NULL|
|    2|    6|    NULL|
|    2|   11|    NULL|
|    2|    2|    NULL|
+-----+-----+--------+

+-----+-----+--------+
|label|value|squared3|
+-----+-----+--------+
|    1|  0.0|    NULL|
|    1|  8.0|    NULL|
|    1|  3.0|    NULL|
|    1|  3.0|    NULL|
|    2|  1.0|    NULL|
|    2|  9.0|    NULL|
|    2|  6.0|    NULL|
|    2|  6.0|    NULL|
|    2| 11.0|    NULL|
|    2|  2.0|    NULL|
+-----+-----+--------+

+-----+-----+--------+
|label|v

### 1.6 Handling null check

UDF's are error-prone when not designed carefully. for example, when you have a column that contains the value null on some records

Below points to remember

Its always best practice to check for null inside a UDF function rather than checking for null outside.
In any case, if you can't do a null check in UDF at lease use IF or CASE WHEN to check for null and call UDF conditionally.

In [11]:
# Les 2 dernières valeurs sont None
# DataFrame avec des value int
l1 = [(1, 0), (1, 8), (1, 3), (1, 3), (2, 1), (2, 9), (2, 6),\
      (2, 6), (2, None), (2, None)]
df1 = spark.createDataFrame(l1,['label', 'value'])
# Afficher le dataframe
df1.printSchema()
df1.show()

# DataFrame avec des value double
l2 = [(1, 0.), (1, 8.), (1, 3.), (1, 3.), (2, 1.), (2, 9.),\
      (2, 6.), (2, 6.), (2, None), (2, None)]
df2 = spark.createDataFrame(l2,['label', 'value'])
# Afficher le dataframe
df2.printSchema()
df2.show()

# gestion des null
spark.udf.register("_nullsafeSquareUDF_int", lambda z: squared2(z) if not z is None else "" , LongType())

df1.createOrReplaceTempView("VAL_TABLE_1")
df2.createOrReplaceTempView("VAL_TABLE_2")

# test avec gestion des null
spark.sql("select label, value, _nullsafeSquareUDF_int(value) as Name from VAL_TABLE_1") \
     .show()

spark.sql("select label, value, _nullsafeSquareUDF_int(value) as Name from VAL_TABLE_2") \
     .show()

# test sans gestion des null
# La commande suivante déclanche une erreur
#spark.sql("select v2, value, square2r(value) as Name from VAL_TABLE_1") \
#     .show()

root
 |-- label: long (nullable = true)
 |-- value: long (nullable = true)

+-----+-----+
|label|value|
+-----+-----+
|    1|    0|
|    1|    8|
|    1|    3|
|    1|    3|
|    2|    1|
|    2|    9|
|    2|    6|
|    2|    6|
|    2| NULL|
|    2| NULL|
+-----+-----+

root
 |-- label: long (nullable = true)
 |-- value: double (nullable = true)

+-----+-----+
|label|value|
+-----+-----+
|    1|  0.0|
|    1|  8.0|
|    1|  3.0|
|    1|  3.0|
|    2|  1.0|
|    2|  9.0|
|    2|  6.0|
|    2|  6.0|
|    2| NULL|
|    2| NULL|
+-----+-----+

+-----+-----+----+
|label|value|Name|
+-----+-----+----+
|    1|    0|   0|
|    1|    8|  64|
|    1|    3|   9|
|    1|    3|   9|
|    2|    1|   1|
|    2|    9|  81|
|    2|    6|  36|
|    2|    6|  36|
|    2| NULL|NULL|
|    2| NULL|NULL|
+-----+-----+----+

+-----+-----+----+
|label|value|Name|
+-----+-----+----+
|    1|  0.0|NULL|
|    1|  8.0|NULL|
|    1|  3.0|NULL|
|    1|  3.0|NULL|
|    2|  1.0|NULL|
|    2|  9.0|NULL|
|    2|  6.0|N

L'application sur une valeur d'un mauvais type entraine *null* mais l'application sur *null* entraine une erreur (sauf si on le gère).


# 2. Pandas UDF (pandas_udf) 

By using pyspark.sql.functions.pandas_udf() function you can create a Pandas UDF (User Defined Function) that is executed by PySpark with Arrow to transform the DataFrame. PySpark by default provides hundreds of built-in function hence before you create your own function, I would recommend doing little research to identify if the function you are creating is already available in pyspark.sql.functions.

 Syntax
```
pandas_udf(f=None, returnType=None, functionType=None`
```

- f - User defined function
- returnType - This is optional but when specified it should be either a DDL-formatted type string or any type of pyspark.sql.types.DataType
- functionType - int, optional

### PySpark pandas_udf() Usage

The pandas_udf() is a built-in function from pyspark.sql.functions that is used to create the Pandas user-defined function and apply the custom function to a column or to the entire DataFrame.


### 2.1 Les *import*


In [12]:
# Imports 
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import *

import pandas as pd

### 2.2 Creating a data frame

- create 2 data frames with 2 columns (label, value)

The difference between the 2 dataframes is that for the first, the "value" values are of type long or int, and for the second, the "value" values are of type double.

The label can take 2 values and appears in at least 4 lines.

In [13]:
# DataFrame avec des value int
l1 = [(1, 0), (1, 8), (1, 3), (1, 3), (2, 1), (2, 9), (2, 6), (2, 6), (2, 11), (2, 2)]
df1 = spark.createDataFrame(l1,['label', 'value'])
# Display Schema and dataframe
df1.printSchema()
df1.show()

# DataFrame avec des value double
l2 = [(1, 0.), (1, 8.), (1, 3.), (1, 3.), (2, 1.), (2, 9.), (2, 6.), (2, 6.), (2, 11.), (2, 2.)]
df2 = spark.createDataFrame(l2,['label', 'value'])
# Display schema and dataframe
df2.printSchema()
df2.show()

root
 |-- label: long (nullable = true)
 |-- value: long (nullable = true)

+-----+-----+
|label|value|
+-----+-----+
|    1|    0|
|    1|    8|
|    1|    3|
|    1|    3|
|    2|    1|
|    2|    9|
|    2|    6|
|    2|    6|
|    2|   11|
|    2|    2|
+-----+-----+

root
 |-- label: long (nullable = true)
 |-- value: double (nullable = true)

+-----+-----+
|label|value|
+-----+-----+
|    1|  0.0|
|    1|  8.0|
|    1|  3.0|
|    1|  3.0|
|    2|  1.0|
|    2|  9.0|
|    2|  6.0|
|    2|  6.0|
|    2| 11.0|
|    2|  2.0|
+-----+-----+



### 2.3 Create PySpark Pandas UDF

By using pandas_udf() let's create the custom UDF function.

The Python function should take a pandas Series as an input and return a pandas Series of the same length, and you should specify these in the Python type hints. Spark runs a pandas UDF by splitting columns into batches, calling the function for each batch as a subset of the data, then concatenating the results.
    

In [14]:
# Declare the function and create the UDF
# 
# Create pandas_udf()
# Method 1 : using decorator
@pandas_udf(IntegerType())
#@pandas_udf('integer', PandasUDFType.SCALAR) # before Spark 3.0
def to_square_int(s: pd.Series) -> pd.Series:
    return s*s

@pandas_udf(FloatType())
#@pandas_udf('integer', PandasUDFType.SCALAR) # before Spark 3.0
def to_square_float(s: pd.Series) -> pd.Series:
    return s*s

# Method 2 : using pandas_udf function
def to_square1(s: pd.Series) -> pd.Series:
    return s*s
to_square_udf = pandas_udf(to_square1, returnType=IntegerType())


# Using UDF with select()
df1.withColumn("square",to_square_int(df1.value)).show()
df1.withColumn("square",to_square_float(df1.value)).show()

df2.withColumn("square",to_square_int(df2.value)).show()
df2.withColumn("square",to_square_float(df2.value)).show()

df1.select(df1.label,df1.value,to_square_udf(df1.value).alias("squared")).show()

+-----+-----+------+
|label|value|square|
+-----+-----+------+
|    1|    0|     0|
|    1|    8|    64|
|    1|    3|     9|
|    1|    3|     9|
|    2|    1|     1|
|    2|    9|    81|
|    2|    6|    36|
|    2|    6|    36|
|    2|   11|   121|
|    2|    2|     4|
+-----+-----+------+

+-----+-----+------+
|label|value|square|
+-----+-----+------+
|    1|    0|   0.0|
|    1|    8|  64.0|
|    1|    3|   9.0|
|    1|    3|   9.0|
|    2|    1|   1.0|
|    2|    9|  81.0|
|    2|    6|  36.0|
|    2|    6|  36.0|
|    2|   11| 121.0|
|    2|    2|   4.0|
+-----+-----+------+

+-----+-----+------+
|label|value|square|
+-----+-----+------+
|    1|  0.0|     0|
|    1|  8.0|    64|
|    1|  3.0|     9|
|    1|  3.0|     9|
|    2|  1.0|     1|
|    2|  9.0|    81|
|    2|  6.0|    36|
|    2|  6.0|    36|
|    2| 11.0|   121|
|    2|  2.0|     4|
+-----+-----+------+

+-----+-----+------+
|label|value|square|
+-----+-----+------+
|    1|  0.0|   0.0|
|    1|  8.0|  64.0|
|    1|  3

In [15]:
from scipy import stats

@pandas_udf('double')
def cdf(v):
    return pd.Series(stats.norm.cdf(v/10))

df1.withColumn('cumulative_probability', cdf(df1.value)).show()

+-----+-----+----------------------+
|label|value|cumulative_probability|
+-----+-----+----------------------+
|    1|    0|                   0.5|
|    1|    8|    0.7881446014166034|
|    1|    3|    0.6179114221889526|
|    1|    3|    0.6179114221889526|
|    2|    1|     0.539827837277029|
|    2|    9|    0.8159398746532405|
|    2|    6|    0.7257468822499265|
|    2|    6|    0.7257468822499265|
|    2|   11|    0.8643339390536173|
|    2|    2|     0.579259709439103|
+-----+-----+----------------------+



we subtract mean of v from each value of v for each group. 

The grouping function is defined by the "groupby" function, i.e, each input pandas.

DataFrame to the user-defined function has the same "label" value. 

The input and output schema of this user-defined function are the same, so we pass "df.schema" to the decorator pandas_udf for specifying the schema.

In [16]:
print("df1")
df1.show()
print("df2")
df2.show()

print("df1 : mean1")

@pandas_udf(df1.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def mean1(pdf):
    return pdf.assign(value=pdf.value.mean())

# apply on df1
print(" mean1 on df1")
df1.groupby('label').apply(mean1).show()
# apply on df2
print(" mean1 on df2")
df2.groupby('label').apply(mean1).show()


print("df1 : substract mean1")

@pandas_udf(df1.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def subtract_mean1(pdf):
    return pdf.assign(value=pdf.value - pdf.value.mean())

df1.groupby('label').apply(subtract_mean1).show()
df2.groupby('label').apply(mean1).show()
print("df2")
df2.show()

print("df2 : mean")

@pandas_udf(df2.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def mean2(pdf):
    return pdf.assign(value=pdf.value.mean())
print(" df2 : mean2 on df1")
df1.groupby('label').apply(mean2).show()
print(" df2 : mean2 on df2")
df2.groupby('label').apply(mean2).show()

print("df2 : substract mean2")

@pandas_udf(df2.schema, PandasUDFType.GROUPED_MAP)
# Input/output are both a pandas.DataFrame
def subtract_mean2(pdf):
    return pdf.assign(value=pdf.value - pdf.value.mean())

df2.groupby('label').apply(subtract_mean2).show()



df1
+-----+-----+
|label|value|
+-----+-----+
|    1|    0|
|    1|    8|
|    1|    3|
|    1|    3|
|    2|    1|
|    2|    9|
|    2|    6|
|    2|    6|
|    2|   11|
|    2|    2|
+-----+-----+

df2
+-----+-----+
|label|value|
+-----+-----+
|    1|  0.0|
|    1|  8.0|
|    1|  3.0|
|    1|  3.0|
|    2|  1.0|
|    2|  9.0|
|    2|  6.0|
|    2|  6.0|
|    2| 11.0|
|    2|  2.0|
+-----+-----+

df1 : mean1
 mean1 on df1


/usr/local/spark/python/pyspark/sql/pandas/group_ops.py:122: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


+-----+-----+
|label|value|
+-----+-----+
|    1|    3|
|    1|    3|
|    1|    3|
|    1|    3|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
+-----+-----+

 mean1 on df2
+-----+-----+
|label|value|
+-----+-----+
|    1|    3|
|    1|    3|
|    1|    3|
|    1|    3|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
+-----+-----+

df1 : substract mean1
+-----+-----+
|label|value|
+-----+-----+
|    1|   -3|
|    1|    4|
|    1|    0|
|    1|    0|
|    2|   -4|
|    2|    3|
|    2|    0|
|    2|    0|
|    2|    5|
|    2|   -3|
+-----+-----+

+-----+-----+
|label|value|
+-----+-----+
|    1|    3|
|    1|    3|
|    1|    3|
|    1|    3|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
|    2|    5|
+-----+-----+

df2
+-----+-----+
|label|value|
+-----+-----+
|    1|  0.0|
|    1|  8.0|
|    1|  3.0|
|    1|  3.0|
|    2|  1.0|
|    2|  9.0|
|    2|  6.0|
|    2|  6.0|
|    2| 11.0|
|   

- Explain the previous result (df1 and df2 have the same values ?)


# 3. time testing

- create a dataframe with range (10000 rows)

square function exits in spark.

Compare time excecution between square spark function , udf python square function, pandas udf python square function

In [17]:
import time
from pyspark.mllib.random import RandomRDDs
from pyspark.sql.functions import pow

@udf("float")
def square4(s):
  return s * s

@pandas_udf(FloatType())
#@pandas_udf('integer', PandasUDFType.SCALAR) # before Spark 3.0
def to_square_float(s: pd.Series) -> pd.Series:
    return s*s

nbre = 100000000
seconds1 = time.time()
rdd1 = RandomRDDs.normalRDD(sc, nbre, seed=1).map(lambda x: x).zipWithIndex()
df1 = spark.createDataFrame(rdd1,["value"])

df1 = df1.withColumn("squared",square4(col("value")))
df1.show()
seconds2 = time.time()
print("==========\n temps écoulé : ", seconds2-seconds1)
print("==========\n")

rdd2 = RandomRDDs.normalRDD(sc, nbre, seed=1).map(lambda x: x).zipWithIndex()
df2 = spark.createDataFrame(rdd2,["value"])

df2 = df2.withColumn("squared",to_square_float(col("value")))
df2.show()
seconds3 = time.time()
print("==========\n temps écoulé : ", seconds3-seconds2)
print("==========\n")

rdd3 = RandomRDDs.normalRDD(sc, nbre, seed=1).map(lambda x: x).zipWithIndex()
df3 = spark.createDataFrame(rdd3,["value"])

df3 = df3.withColumn("squared",pow(col("value"),2))
df3.show()
seconds4 = time.time()
print("==========\n temps écoulé : ", seconds4-seconds3)
print("==========\n")

+--------------------+---+-----------+
|               value| _2|    squared|
+--------------------+---+-----------+
|  0.2901957375442786|  0| 0.08421357|
| 0.11165831824048798|  1| 0.01246758|
| -0.2667525536068808|  2| 0.07115693|
| -1.2246814950366627|  3|  1.4998448|
|  -0.211734048360863|  4|0.044831306|
|-0.34760985561390856|  5|0.120832615|
|  2.9765559506194617|  6|   8.859885|
|  1.0088848296456696|  7|  1.0178486|
|  1.4814327708727246|  8|   2.194643|
|  0.7796065099600471|  9|  0.6077863|
|-0.28564589998339107| 10| 0.08159358|
| -0.6557637363792528| 11| 0.43002608|
|  0.5063168531583091| 12| 0.25635675|
|  1.9214560883356222| 13|  3.6919935|
|  0.0728082272048009| 14|0.005301038|
|   2.206764434538693| 15|   4.869809|
| -1.4536446890283143| 16|   2.113083|
|  1.3570054083741572| 17|  1.8414637|
|  0.3012170908743003| 18| 0.09073173|
| -1.2988156552131784| 19|  1.6869221|
+--------------------+---+-----------+
only showing top 20 rows
 temps écoulé :  3.828939199447632

+--

Faster or not Faster ?

How can we measure the performance ?

Who is the fastest (and how) ?

In [18]:
sc.stop()